# Rainfall Analysis — Five-County Planting Advisory
**Course:** PLP Data Analytics · Assignment 2  
**Dataset:** Monthly rainfall (mm) for Nairobi, Nakuru, Kisumu, Mombasa, and Machakos

In [1]:
import numpy as np
import csv

## 1. Raw Data

In [2]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ", "9", "12", "", "45", "182", "96"]

MONTHS = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

## 2. Data Cleaning — Machakos

In [3]:
def clean_machakos(raw):
    """Parse a list of raw string values to integers.

    Valid numeric strings are converted with int().
    Empty strings, 'N/A', or any non-numeric token raise ValueError
    and are recorded for later imputation.
    """
    parsed = []
    invalid_indices = []

    for i, token in enumerate(raw):
        try:
            value = int(token.strip())   # strip whitespace then cast
            if value < 0:
                raise ValueError(f"Negative rainfall ({value}) is not physical")
            parsed.append(value)
        except ValueError:
            parsed.append(None)
            invalid_indices.append(i)

    # Impute: replace None with the rounded mean of valid values
    valid_values = [v for v in parsed if v is not None]
    imputed_value = round(sum(valid_values) / len(valid_values))

    for idx in invalid_indices:
        parsed[idx] = imputed_value
        print(f"  Imputed {MONTHS[idx]:>10s} (index {idx}): {imputed_value} mm "
              f"(original: {repr(raw[idx])})")

    return parsed


print("Machakos cleaning — imputed months:")
machakos_clean = clean_machakos(machakos_raw)
print()
print("Cleaned Machakos row:", machakos_clean)

Machakos cleaning — imputed months:
  Imputed      April (index 3): 68 mm (original: 'N/A')
  Imputed  September (index 8): 68 mm (original: '')

Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 3. Combined 5 × 12 NumPy Array

In [4]:
all_rows = clean_rows + [machakos_clean]
R = np.array(all_rows, dtype=int)

print("Shape :", R.shape)
print("dtype :", R.dtype)
print()
print(R)

Shape : (5, 12)
dtype : int64

[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## 4. Annual Totals and Monthly Averages

In [5]:
annual_totals   = R.sum(axis=1)     # sum across columns → one value per county
monthly_averages = R.mean(axis=0)   # mean across rows   → one value per month

print("Annual total rainfall (mm) per county:")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<10s}: {total} mm")

print()
print("Average rainfall (mm) per month across all counties:")
for month, avg in zip(MONTHS, monthly_averages):
    print(f"  {month:<12s}: {avg:.1f} mm")

Annual total rainfall (mm) per county:
  Nairobi   : 1045 mm
  Nakuru    : 916 mm
  Kisumu    : 1371 mm
  Mombasa   : 1075 mm
  Machakos  : 811 mm

Average rainfall (mm) per month across all counties:
  January     : 47.0 mm
  February    : 47.6 mm
  March       : 91.0 mm
  April       : 164.4 mm
  May         : 171.2 mm
  June        : 67.6 mm
  July        : 56.6 mm
  August      : 58.2 mm
  September   : 64.2 mm
  October     : 67.8 mm
  November    : 127.4 mm
  December    : 80.6 mm


## 5. Wettest Month per County (np.argmax)

In [6]:
wettest_indices = np.argmax(R, axis=1)          # index of max along each row
wettest_months  = [MONTHS[i] for i in wettest_indices]

print("Wettest month per county:")
for county, month, c_idx in zip(counties, wettest_months, range(len(counties))):
    idx = wettest_indices[c_idx]
    print(f"  {county:<10s}: {month} ({R[c_idx, idx]} mm)")

Wettest month per county:
  Nairobi   : April (219 mm)
  Nakuru    : April (145 mm)
  Kisumu    : April (240 mm)
  Mombasa   : May (240 mm)
  Machakos  : November (182 mm)


## 6. Dry-Month Counts (below 50 mm)

In [7]:
dry_mask   = R < 50                   # boolean array — True where rainfall < 50 mm
dry_counts = dry_mask.sum(axis=1)     # count Trues along each row

print("Dry months (< 50 mm) per county:")
for county, count in zip(counties, dry_counts):
    print(f"  {county:<10s}: {count} month(s)")

Dry months (< 50 mm) per county:
  Nairobi   : 4 month(s)
  Nakuru    : 3 month(s)
  Kisumu    : 0 month(s)
  Mombasa   : 2 month(s)
  Machakos  : 6 month(s)


## 7. Planting Windows — Two Consecutive Wettest Months

In [8]:
# Shifted-array technique: sum each pair of adjacent months
pair_sums     = R[:, :-1] + R[:, 1:]           # shape (5, 11)
best_pair_idx = np.argmax(pair_sums, axis=1)   # starting month index per county

planting_windows = [
    f"{MONTHS[i]}\u2013{MONTHS[i+1]}"
    for i in best_pair_idx
]

print("Recommended planting window (two consecutive wettest months):")
for county, window, c_idx in zip(counties, planting_windows, range(len(counties))):
    combined = pair_sums[c_idx, best_pair_idx[c_idx]]
    print(f"  {county:<10s}: {window}  ({combined} mm combined)")

Recommended planting window (two consecutive wettest months):
  Nairobi   : April–May  (395 mm combined)
  Nakuru    : April–May  (265 mm combined)
  Kisumu    : April–May  (420 mm combined)
  Mombasa   : April–May  (390 mm combined)
  Machakos  : November–December  (278 mm combined)


## 8. Write rainfall_summary.csv

In [9]:
csv_path = "rainfall_summary.csv"

with open(csv_path, "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    for row in zip(counties, annual_totals, wettest_months, dry_counts, planting_windows):
        writer.writerow(row)

print(f"Wrote '{csv_path}'")
print()

# Verify — read back and display
with open(csv_path, newline="") as fh:
    for line in fh:
        print(line, end="")

Wrote 'rainfall_summary.csv'

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,April,4,April–May
Nakuru,916,April,3,April–May
Kisumu,1371,April,0,April–May
Mombasa,1075,May,2,April–May
Machakos,811,November,6,November–December


## Planting Advisory

Nairobi, Nakuru, Kisumu, and Mombasa all peak in the **April–May** long-rains window (265–420 mm combined), which is the safest and most productive planting period for staple crops such as maize and beans across those four counties. Machakos follows a different rainfall calendar — its wettest consecutive months are **November–December** (278 mm combined), so farmers there should time planting to coincide with the short-to-long-rains transition rather than the main long rains. Kisumu records no dry months at all, while Machakos has six months below 50 mm; NGO advisories should accordingly recommend drought-tolerant varieties and water-harvesting techniques for Machakos and similarly semi-arid areas. A secondary short-rains window in **October–November** exists for Nairobi and Mombasa and can support fast-maturing legumes such as green grams or cowpeas.

**Data-quality caveat:** Two Machakos readings were missing or invalid in the original dataset — `"N/A"` for April and an empty string for September — and were replaced with the rounded mean of the ten valid station readings (68 mm each). The June entry (`" 20 "`) contained only whitespace padding and was successfully parsed after stripping. Any advisory figures for Machakos should be verified against updated station records before dissemination.

## Data Quality Notes

| Issue | Month affected | Raw value | Action taken |
|---|---|---|---|
| Non-numeric string | April | `"N/A"` | Caught by `except ValueError`; replaced with rounded mean of valid values (68 mm) |
| Empty string | September | `""` | Caught by `except ValueError`; replaced with rounded mean of valid values (68 mm) |
| Whitespace padding | June | `" 20 "` | `token.strip()` removed spaces; parsed successfully as 20 mm — no imputation needed |

All cleaning is handled inside `clean_machakos()` via a `try/except ValueError` block. The imputation mean is calculated from valid values within the same row, keeping estimates grounded in Machakos-specific rainfall patterns. Future uploads in the same raw-text format will be cleaned automatically by the same function.